# Sixteen Thousand Parameters That Buy Nothing

**Topic:** Transfer learning &nbsp;·&nbsp; **Difficulty:** Medium

Before you fine-tune, ask what the extra parameters can express that the frozen
model cannot. Here the answer is nothing: a linear adapter feeding a linear head
composes to a single linear map, which is exactly what the probe already fits. The
adapter adds 16,384 parameters and zero representational power.

Measure it across training-set sizes. The two curves should lie on top of
each other, which is the point: capacity you cannot use is not free, it is
just slower.

## The data (created in the next cell)

- `feats_train` `(2000, 128)`: frozen pretrained features, and `y_train`.
- `feats_test` `(1000, 128)`, `y_test`.
- `sizes = [20, 50, 100, 300, 1000, 2000]`.
- The "fine-tuned" model adds a `128 x 128` linear adapter beneath the head,
  initialized at the identity and trained jointly.

## Your task

With **NumPy only**, for each training-set size (take the first `m` rows):

1. `probe`: fit multinomial logistic regression on the frozen features by
   full-batch gradient descent, 400 steps, lr 0.5, weight init zeros.
2. `finetune`: jointly train the same head **and** a `(128, 128)` adapter
   `A` (initialized to the identity), features become `feats @ A`, same 400
   steps and lr.
3. Record test accuracy for both at each size.
4. Set `answers` to a dict with `"acc_probe"`, `"acc_finetune"` (length-6 arrays),
   `"max_abs_gap"` (the largest absolute difference between the two curves),
   `"param_ratio"`, the adapter model's parameter count divided by the
   probe's (count `d*C + C` for the probe and `d*C + C + d*d` for the
   adapter model).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the two curves stay within 3 accuracy points of each other everywhere,
despite a 26x parameter difference.

In [ ]:
import numpy as np

rng = np.random.default_rng(3104)
n_tr, n_te, d, C = 2000, 1000, 128, 5
W_true = rng.normal(0, 1, (d, C))

def make(n, seed):
    r = np.random.default_rng(seed)
    F = r.normal(0, 1, (n, d))
    logits = F @ W_true * 0.28
    p = np.exp(logits - logits.max(axis=1, keepdims=True))
    p /= p.sum(axis=1, keepdims=True)
    lab = np.array([r.choice(C, p=row) for row in p])
    return F, lab

feats_train, y_train = make(n_tr, 101)
feats_test, y_test = make(n_te, 102)
sizes = [20, 50, 100, 300, 1000, 2000]
print(f"features {feats_train.shape}, {C} classes, sweep sizes {sizes}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("acc_probe", [6], 2.84, 1.6908219191677785, 1e-06),
    ("acc_finetune", [6], 2.8000000000000003, 1.642095111309949, 1e-06),
    ("max_abs_gap", [], 0.017000000000000015, -0.0028464462518464884, 1e-06),
    ("param_ratio", [], 26.4015503875969, -4.420623184924196, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "max_abs_gap" in answers:
    _tied = answers["max_abs_gap"] < 0.03
    _cost = answers["param_ratio"] > 26
    print(f"  {'curves coincide':<24} {'ok' if _tied else 'wrong'}  "
          f"(max gap {answers['max_abs_gap']:.4f})")
    print(f"  {'for 26x the parameters':<24} {'ok' if _cost else 'wrong'}  "
          f"(ratio {answers['param_ratio']:.1f}x)")
    _ok = _ok and _tied and _cost

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

`(F @ A) @ W` equals `F @ (A @ W)`, and `A @ W` is just some `128 x 5` matrix,
precisely the space the probe already searches. So the adapter cannot reach any
function the probe could not, and the curves stay within about 2 accuracy
points of each other at every training size. Real
fine-tuning helps because it updates a *nonlinear* backbone. Stacking linear
layers is the classic reminder that depth without nonlinearity is not depth. Probe
first, and only reach for fine-tuning when the extra parameters can express
something new.

```python
import numpy as np

def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

def onehot(y, C):
    O = np.zeros((len(y), C))
    O[np.arange(len(y)), y] = 1.0
    return O

def train(F, y, adapt, steps=400, lr=0.5):
    m, d = F.shape
    C = 5
    W = np.zeros((d, C))
    b = np.zeros(C)
    A = np.eye(d)
    Y = onehot(y, C)
    for _ in range(steps):
        H = F @ A if adapt else F
        P = softmax(H @ W + b)
        G = (P - Y) / m
        gW = H.T @ G
        gb = G.sum(axis=0)
        if adapt:
            gA = F.T @ (G @ W.T)
            A = A - lr * gA
        W = W - lr * gW
        b = b - lr * gb
    return W, b, A

acc_probe, acc_ft = [], []
for m in sizes:
    F, y = feats_train[:m], y_train[:m]

    W, b, _ = train(F, y, adapt=False)
    acc_probe.append(float((softmax(feats_test @ W + b).argmax(1) == y_test).mean()))

    W, b, A = train(F, y, adapt=True)
    acc_ft.append(float((softmax((feats_test @ A) @ W + b).argmax(1) == y_test).mean()))

acc_probe = np.array(acc_probe)
acc_ft = np.array(acc_ft)

d_, C_ = feats_train.shape[1], 5
probe_params = d_ * C_ + C_
adapter_params = probe_params + d_ * d_

answers = {
    "acc_probe": acc_probe,
    "acc_finetune": acc_ft,
    "max_abs_gap": float(np.abs(acc_probe - acc_ft).max()),
    "param_ratio": float(adapter_params / probe_params),
}
```